# 01. 우편번호 가공 노트북 v1 (선택 지역)

**목적**: 우체국 우편번호 DB(건물 단위 약 30만 행)와 국가기초구역 도형을 **우편번호 1건 = 1레코드**로 요약해,
사이트(Astro)가 바로 읽는 정적 JSON과 점검용 엑셀을 만든다.

**입력** (`data/raw/`, 저장소에는 올리지 않음)
- `zipcode_DB/` 지역별 주소 DB (시도별 txt, UTF-8, `|` 구분)
- `rangeaddr/`, `pobox/`, `chgaddr/` 범위·사서함·변경분 (대조용)
- `zone/` 국가기초구역 SHP (`TL_KODIS_BAS_{시도코드}_{YYYYMM}.shp`, EPSG:5179, cp949)

**출력** (`data/processed/`)
- `meta.json`, `zip_index.json`(검색 색인), `hierarchy.json`, `emd.json`, `zips/{우편번호}.json`, `geo/{시군구코드}.geojson`
- `qa/zipcode_qa_v1.xlsx` (종합판정, 개수·집합 대조, 걸침, 폴리곤 품질, 우편번호별 요약, 데이터 한계)

**정확도 등급**: 이 노트북의 모든 집계는 *주소(건물) 기준 집계*이므로 `정확`이다. 통계 배분 값(추정)은 이후 단계에서 별도 등급으로 붙인다.

In [ ]:
import os, re, json, shutil, collections, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import shapely
from shapely import STRtree
from shapely.geometry import mapping
warnings.filterwarnings("ignore", category=FutureWarning)

# ---------------- 설정 ----------------
RAW = Path(os.environ.get("KRPC_RAW", Path.cwd().parent / "raw"))
OUT = Path(os.environ.get("KRPC_OUT", Path.cwd().parent / "processed"))

# 대상 지역. sigungu_prefix: DB '시군구'/도형 'SIG_KOR_NM'이 이 접두어로 시작하는 곳만 사용 (None=시도 전체)
REGIONS = [
    {"key": "sejong",  "sido": "세종특별자치시", "db": "세종특별자치시.txt", "shp": "36", "sigungu_prefix": None},
    {"key": "daejeon", "sido": "대전광역시",     "db": "대전광역시.txt",     "shp": "30", "sigungu_prefix": None},
    {"key": "gyeonggi3", "sido": "경기도",       "db": "경기도.txt",         "shp": "41", "sigungu_prefix": ["성남시", "수원시", "용인시"]},
]

NOTICE_DATE = "2026-10-13"      # 고시일자
BASE_DATE = "2026-10-08"        # 기준일 (우체국 안내문)
ZONE_VER = "202610"             # 기초구역 도형 버전
SIMPLIFY_M = 2.0                # 폴리곤 단순화 허용오차 (m, EPSG:5179)
COORD_DEC = 5                   # 좌표 소수 자릿수 (약 1.1 m)
TOP_ROADS = 8
OVERLAP_MIN_M2 = 100

# 점검 기준 (임의의 1차 기준: 실제 데이터를 본 뒤 조정)
TH = {"set_match_min": 0.99, "dup_max": 0, "invalid_max": 0, "overlap_pairs_max": 0, "area_dev_max": 0.05}

print("RAW =", RAW.resolve()); print("OUT =", OUT.resolve())
assert (RAW / "zipcode_DB").exists(), "RAW/zipcode_DB 폴더가 없습니다 (KRPC_RAW 환경변수 또는 data/raw 위치 확인)"

## 1. 우편번호 DB 읽기 (지역별 주소 DB)
한 줄 = 건물(주소) 1건. 필요한 열만 읽는다. 지역 이름 규칙:
- **시군구**: DB `시군구` 열. 세종은 비어 있어 시도명을 쓴다. 시군구 코드는 `법정동코드` 앞 5자리.
- **읍면동(법정동 단위)**: `법정동명`이 있으면 그것, 없으면 `읍면`+`리명`. 코드는 `법정동코드` 10자리.

In [ ]:
DB_COLS = ["우편번호", "시도", "시군구", "읍면", "도로명코드", "도로명", "건물관리번호",
           "법정동코드", "법정동명", "리명", "행정동명"]

def read_db(region):
    p = RAW / "zipcode_DB" / region["db"]
    df = pd.read_csv(p, sep="|", encoding="utf-8-sig", dtype=str, usecols=DB_COLS, keep_default_na=False)
    n_file = len(df)
    if region["sigungu_prefix"]:
        df = df[df["시군구"].str.startswith(tuple(region["sigungu_prefix"]))].copy()
    df["region"] = region["key"]
    return df, n_file

frames, file_rows = [], {}
for r in REGIONS:
    d, n = read_db(r); frames.append(d); file_rows[r["key"]] = n
    print(f'{r["key"]:10s} 파일 {n:>9,d}행 → 대상 {len(d):>9,d}행, 우편번호 {d["우편번호"].nunique():,d}개')
db = pd.concat(frames, ignore_index=True)

db["sigungu"] = np.where(db["시군구"] != "", db["시군구"], db["시도"])
db["sigungu_code"] = db["법정동코드"].str[:5]
db["emd_name"] = np.where(db["법정동명"] != "", db["법정동명"], (db["읍면"] + " " + db["리명"]).str.strip())
db["emd_code"] = db["법정동코드"]
HD_BLANK = "(미표기)"   # 행정동명이 빈 행: 우체국 명세서에 의미 설명이 없어 법정동과 같다고 가정하지 않고 따로 둔다
db["hdong"] = np.where(db["행정동명"] != "", db["행정동명"], HD_BLANK)
hd_blank_rows = db.groupby("region")["hdong"].apply(lambda s: float((s == HD_BLANK).mean())).round(4).to_dict()
print("행정동명 미표기 행 비율:", hd_blank_rows)
assert (db["우편번호"].str.fullmatch(r"\d{5}")).all(), "5자리 아닌 우편번호 존재"
assert (db["법정동코드"].str.fullmatch(r"\d{10}")).all(), "법정동코드 10자리 아님"
print("총", f"{len(db):,d}", "행 / 우편번호", db["우편번호"].nunique())
dup_bldg = int(db["건물관리번호"].duplicated().sum()); print("건물관리번호 중복:", dup_bldg)

## 2. 우편번호별 집계
비율은 모두 *주소(건물) 건수 기준*이다. 한 우편번호가 여러 읍면동·행정동에 걸치면 각각의 비율을 계산한다.

In [ ]:
def shares(df, keys, extra=None):
    g = df.groupby(["우편번호"] + keys, sort=False).size().rename("n").reset_index()
    g["share"] = g["n"] / g.groupby("우편번호")["n"].transform("sum")
    return g.sort_values(["우편번호", "n"], ascending=[True, False])

sg = shares(db, ["sigungu_code", "sigungu"])
emd = shares(db, ["emd_code", "emd_name"])
hd = shares(db, ["hdong"])
# 읍면동별 대표 행정동 (건수 최다)
emd_h = (db.groupby(["우편번호", "emd_code", "hdong"]).size().rename("n").reset_index()
           .sort_values(["우편번호", "emd_code", "n"], ascending=[True, True, False]))
roads = (db.groupby(["우편번호", "도로명"]).size().rename("n").reset_index()
           .sort_values(["우편번호", "n"], ascending=[True, False]))
n_road_code = db.groupby("우편번호")["도로명코드"].nunique().rename("n_road")
n_addr = db.groupby("우편번호").size().rename("n_addr")
sido_of = db.groupby("우편번호")["시도"].agg(lambda s: s.value_counts().index[0]).rename("sido")

summary = pd.concat([sido_of, n_addr, n_road_code], axis=1)
summary["n_sigungu"] = sg.groupby("우편번호").size()
summary["n_emd"] = emd.groupby("우편번호").size()
summary["n_hdong"] = hd[hd["hdong"] != HD_BLANK].groupby("우편번호").size().reindex(summary.index).fillna(0).astype(int)  # 미표기는 걸침 계산에서 제외
summary["hdong_blank_share"] = hd[hd["hdong"] == HD_BLANK].set_index("우편번호")["share"].reindex(summary.index).fillna(0).round(4)
top_sg = sg.groupby("우편번호").first()
top_emd = emd.groupby("우편번호").first()
summary["primary_sigungu_code"] = top_sg["sigungu_code"]
summary["primary_sigungu"] = top_sg["sigungu"]
summary["primary_sigungu_share"] = top_sg["share"].round(4)
summary["primary_emd"] = top_emd["emd_name"]
summary["primary_emd_share"] = top_emd["share"].round(4)
summary = summary.reset_index().rename(columns={"index": "zip", "우편번호": "zip"})
print(summary.head(3).to_string())
print("시군구 걸침 번호:", int((summary["n_sigungu"] > 1).sum()), "/ 읍면동 걸침:", int((summary["n_emd"] > 1).sum()),
      "/ 행정동 걸침:", int((summary["n_hdong"] > 1).sum()))

## 3. 기초구역 도형 읽기·정리
EPSG:5179(미터)에서 면적·겹침을 계산하고, 사이트용은 WGS84(EPSG:4326)로 변환한다. 단순화는 우편번호별로 따로 하므로 이웃 경계에 미세한 틈·겹침이 생길 수 있다(현 단계의 한계).

In [ ]:
def read_zone(region):
    code = region["shp"]
    cand = sorted(RAW.rglob(f"TL_KODIS_BAS_{code}_*.shp"))  # data/raw/zone/ 아래 (하위 폴더 무관)
    assert cand, f"SHP 없음: TL_KODIS_BAS_{code}_*.shp"
    g = gpd.read_file(cand[-1], encoding="cp949")
    n_file = len(g)
    if region["sigungu_prefix"]:
        g = g[g["SIG_KOR_NM"].fillna("").str.startswith(tuple(region["sigungu_prefix"]))].copy()
    g["region"] = region["key"]
    return g, n_file, cand[-1].name

zs, zone_rows, zone_files = [], {}, {}
for r in REGIONS:
    g, n, fn = read_zone(r); zs.append(g); zone_rows[r["key"]] = n; zone_files[r["key"]] = fn
    print(f'{r["key"]:10s} {fn}  파일 {n:,d} → 대상 {len(g):,d}  CRS={g.crs.to_string()}')
zone = pd.concat(zs, ignore_index=True)
zone = gpd.GeoDataFrame(zone, geometry="geometry", crs="EPSG:5179")
zone = zone.rename(columns={"BAS_ID": "zip"})
zone["area_km2_calc"] = zone.geometry.area / 1e6
zone["area_dev"] = (zone["area_km2_calc"] - zone["BAS_AR"]).abs() / zone["BAS_AR"].replace(0, np.nan)

# 품질 검사
q_dup = zone["zip"].duplicated().sum()
q_invalid = int((~zone.geometry.is_valid).sum()); q_empty = int(zone.geometry.is_empty.sum())
geoms = zone.geometry.values; tree = STRtree(geoms)
i_idx, j_idx = tree.query(geoms, predicate="intersects")
ov_rows = []
for i, j in zip(i_idx, j_idx):
    if i < j:
        a = geoms[i].intersection(geoms[j]).area
        if a > OVERLAP_MIN_M2:
            ov_rows.append({"zip_a": zone["zip"].iat[i], "zip_b": zone["zip"].iat[j], "겹침_m2": round(a, 1)})
overlaps = pd.DataFrame(ov_rows, columns=["zip_a", "zip_b", "겹침_m2"])
print("중복 ID:", q_dup, "/ 무효:", q_invalid, "/ 빈 도형:", q_empty, "/ 겹침 쌍(>%dm²):" % OVERLAP_MIN_M2, len(overlaps))
print("면적 편차(계산 vs BAS_AR) 최대:", round(float(zone["area_dev"].max()), 4))

## 4. DB ↔ 도형 대조 (개수·집합)
DB에만 있는 번호는 지도에 그릴 수 없고, 도형에만 있는 번호는 주소가 없는 구역(원인은 별도 확인 필요)이다.

In [ ]:
db_set = set(summary["zip"]); sh_set = set(zone["zip"])
reg_of_zip = pd.concat([db[["우편번호", "region"]].drop_duplicates().rename(columns={"우편번호": "zip"}),
                        zone[["zip", "region"]]]).drop_duplicates("zip").set_index("zip")["region"]
cmp_rows = []
for r in REGIONS:
    k = r["key"]
    a = set(summary.loc[summary["zip"].map(reg_of_zip) == k, "zip"])
    b = set(zone.loc[zone["region"] == k, "zip"])
    cmp_rows.append({"지역": k, "DB 우편번호 수": len(a), "도형 수": len(b), "교집합": len(a & b),
                     "DB에만": len(a - b), "도형에만": len(b - a),
                     "DB 기준 일치율": round(len(a & b) / len(a), 4) if a else np.nan})
cmp_df = pd.DataFrame(cmp_rows); print(cmp_df.to_string(index=False))
only_poly = zone[~zone["zip"].isin(db_set)][["zip", "region", "CTP_KOR_NM", "SIG_KOR_NM", "BAS_AR", "MVMN_DE", "MVMN_RESN"]].copy()
only_db = sorted(db_set - sh_set)

# 범위주소/사서함/변경분 대조 (있으면)
def col0(path, enc="utf-8-sig"):
    return pd.read_csv(path, sep="|", encoding=enc, dtype=str, keep_default_na=False, usecols=[0, 1]).iloc[:, :2]

aux_rows = []
all_zip = db_set | sh_set
sido_names = {r["sido"] for r in REGIONS}
for label, files in [("도로명범위", ["rangeaddr/20261013_도로명범위.txt"]), ("지번범위", ["rangeaddr/20261013_지번범위.txt"]),
                     ("사서함", ["pobox/20261013_사서함.txt"]), ("변경분", ["chgaddr/20261013_변경분.txt"])]:
    p = RAW / files[0]
    if not p.exists():
        aux_rows.append({"파일": label, "상태": "파일 없음"}); continue
    x = col0(p); x.columns = ["zip", "시도"]
    x = x[x["시도"].isin(sido_names)]
    # 경기도는 3개 시만 대상이므로 우편번호 집합으로 한정
    sub = x[(x["시도"] != "경기도") | x["zip"].isin(db_set | sh_set)]
    zs_ = set(sub["zip"])
    aux_rows.append({"파일": label, "대상 지역 행 수": len(sub), "우편번호 수": len(zs_),
                     "본 DB에 없는 번호": len(zs_ - db_set), "본 DB에만 있는 번호": len(db_set - zs_) if label in ("도로명범위", "지번범위") else None})
aux_df = pd.DataFrame(aux_rows); print(aux_df.to_string(index=False))

## 5. 우편번호 마스터 합치기 + 폴리곤 (WGS84, 단순화)

In [ ]:
z4326 = zone.to_crs(4326)
simp = zone.geometry.simplify(SIMPLIFY_M, preserve_topology=True)
simp4326 = gpd.GeoSeries(simp, crs=5179).to_crs(4326)

def round_geom(g):
    g = shapely.transform(g, lambda c: np.round(c, COORD_DEC))
    return g if g.is_valid else shapely.make_valid(g)
simp4326 = simp4326.apply(round_geom)
rep = gpd.GeoSeries(zone.geometry.representative_point(), crs=5179).to_crs(4326)

zone_attr = zone[["zip", "region", "CTP_KOR_NM", "SIG_KOR_NM", "BAS_AR", "area_km2_calc", "MVMN_DE", "MVMN_RESN", "NTFC_DE"]].copy()
zone_attr["geom"] = simp4326.values
zone_attr["center_lon"] = rep.x.values; zone_attr["center_lat"] = rep.y.values
b = z4326.geometry.bounds
zone_attr["bbox"] = list(zip(b.minx.round(COORD_DEC), b.miny.round(COORD_DEC), b.maxx.round(COORD_DEC), b.maxy.round(COORD_DEC)))

master = summary.merge(zone_attr, on="zip", how="outer", indicator=True)
master["has_address"] = master["_merge"].isin(["both", "left_only"])
master["has_polygon"] = master["_merge"].isin(["both", "right_only"])
master["sido"] = master["sido"].fillna(master["CTP_KOR_NM"])
master["region"] = master["region"].fillna(master["zip"].map(reg_of_zip))
master = master.drop(columns="_merge").sort_values("zip").reset_index(drop=True)
print(master[["has_address", "has_polygon"]].value_counts().to_string())

## 6. 출력: 우편번호별 JSON · 색인 · 계층 · 읍면동 · 지도
사이트가 읽는 형식이다. 모든 레코드에 `source`(출처·기준일)와 `grade`(정확도 등급)를 넣는다.

In [ ]:
if OUT.exists():
    for sub in ["zips", "geo", "qa"]:
        shutil.rmtree(OUT / sub, ignore_errors=True)
    for f in ["meta.json", "zip_index.json", "hierarchy.json", "emd.json"]:
        (OUT / f).unlink(missing_ok=True)
for sub in ["zips", "geo", "qa"]:
    (OUT / sub).mkdir(parents=True, exist_ok=True)

SOURCE = {"db": f"우체국 우편번호 DB 지역별 주소 (고시 {NOTICE_DATE}, 기준 {BASE_DATE})",
          "zone": f"국가기초구역 도형 {ZONE_VER}"}

def dump(path, obj):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, separators=(",", ":"))

sg_by = {k: v for k, v in sg.groupby("우편번호")}
emd_by = {k: v for k, v in emd.groupby("우편번호")}
hd_by = {k: v for k, v in hd.groupby("우편번호")}
rd_by = {k: v for k, v in roads.groupby("우편번호")}
emdh_by = {k: v for k, v in emd_h.groupby("우편번호")}

index_rows, geo_by_sg = [], collections.defaultdict(list)
for _, m in master.iterrows():
    z = m["zip"]; rec = {"zip": z, "sido": m["sido"], "grade": "정확", "source": SOURCE,
                         "has_address": bool(m["has_address"]), "has_polygon": bool(m["has_polygon"])}
    if m["has_address"]:
        s = sg_by[z]; e = emd_by[z]; h = hd_by[z]; r = rd_by[z].head(TOP_ROADS)
        eh = emdh_by[z]
        rec["n_addr"] = int(m["n_addr"]); rec["n_road"] = int(m["n_road"])
        rec["sigungu"] = [{"code": a.sigungu_code, "name": a.sigungu, "n": int(a.n), "share": round(float(a.share), 4)} for a in s.itertuples()]
        rec["emd"] = [{"code": a.emd_code, "name": a.emd_name, "n": int(a.n), "share": round(float(a.share), 4),
                       "hdong": list(eh[eh["emd_code"] == a.emd_code]["hdong"].head(3))} for a in e.itertuples()]
        rec["hdong"] = [{"name": a.hdong, "n": int(a.n), "share": round(float(a.share), 4)} for a in h.itertuples()]
        rec["roads"] = [{"name": a.도로명, "n": int(a.n)} for a in r.itertuples()]
        rec["flags"] = {"multi_sigungu": bool(m["n_sigungu"] > 1), "multi_emd": bool(m["n_emd"] > 1),
                        "multi_hdong": bool(m["n_hdong"] > 1), "hdong_blank_share": float(m["hdong_blank_share"]), "primary_emd_share": float(m["primary_emd_share"])}
        sg_code, sg_name = m["primary_sigungu_code"], m["primary_sigungu"]
    else:
        # 주소가 없는 구역: 도형의 시군구(시 단위)만 알 수 있다
        sg_code, sg_name = None, (m["SIG_KOR_NM"] if isinstance(m["SIG_KOR_NM"], str) else m["sido"])
        rec["n_addr"] = 0; rec["note"] = "우편번호 DB에 주소가 없는 구역(도형만 존재)"
    rec["primary_sigungu"] = {"code": sg_code, "name": sg_name}
    if m["has_polygon"]:
        rec["geo"] = {"area_km2": round(float(m["area_km2_calc"]), 4), "center": [round(float(m["center_lon"]), 5), round(float(m["center_lat"]), 5)],
                      "bbox": [float(v) for v in m["bbox"]], "polygon": mapping(m["geom"])}
        rec["zone"] = {"created_or_changed": m["MVMN_DE"], "reason": m["MVMN_RESN"], "notified": m["NTFC_DE"]}
        if sg_code: geo_by_sg[sg_code].append({"type": "Feature", "properties": {"zip": z, "n_addr": rec["n_addr"]}, "geometry": mapping(m["geom"])})
    dump(OUT / "zips" / f"{z}.json", rec)
    tokens = []
    if m["has_address"]:
        tokens = [x["name"] for x in rec["emd"][:6]] + [x["name"] for x in rec["hdong"][:4] if x["name"] != HD_BLANK] + [x["name"] for x in rec["roads"]]
    index_rows.append({"z": z, "s": m["sido"], "g": sg_name, "e": m["primary_emd"] if isinstance(m["primary_emd"], str) else "",
                       "n": rec["n_addr"], "c": rec.get("geo", {}).get("center"), "k": " ".join(dict.fromkeys(tokens))})
dump(OUT / "zip_index.json", index_rows)
for code, feats in geo_by_sg.items():
    dump(OUT / "geo" / f"{code}.geojson", {"type": "FeatureCollection", "features": feats})
print("zips JSON:", len(list((OUT / "zips").glob("*.json"))), " geo:", len(geo_by_sg))

# 계층 (시도 → 시군구 → 우편번호)
hier = collections.OrderedDict()
for (sido, code, name), d in db.groupby(["시도", "sigungu_code", "sigungu"]):
    zips_ = sorted(d["우편번호"].unique())
    hier.setdefault(sido, []).append({"code": code, "name": name, "n_zip": len(zips_), "n_addr": int(len(d)), "zips": zips_})
dump(OUT / "hierarchy.json", [{"sido": k, "sigungu": v} for k, v in hier.items()])

# 읍면동 → 우편번호 (지역 프로필 상품용: 읍면동 기준 역방향)
emd_tbl = (db.groupby(["emd_code", "emd_name", "sigungu_code", "우편번호"]).size().rename("n").reset_index())
emd_out = []
for (code, name, sgc), d in emd_tbl.groupby(["emd_code", "emd_name", "sigungu_code"]):
    tot = int(d["n"].sum())
    emd_out.append({"code": code, "name": name, "sigungu_code": sgc, "n_addr": tot,
                    "zips": [{"zip": a.우편번호, "n": int(a.n), "share_of_emd": round(a.n / tot, 4)} for a in d.sort_values("n", ascending=False).itertuples()]})
dump(OUT / "emd.json", emd_out)

meta = {"built_with": "01_build_subset_v1.ipynb", "notice_date": NOTICE_DATE, "base_date": BASE_DATE, "zone_version": ZONE_VER,
        "regions": [{"key": r["key"], "sido": r["sido"], "sigungu_prefix": r["sigungu_prefix"]} for r in REGIONS],
        "counts": {"zips_total": int(len(master)), "zips_with_address": int(master["has_address"].sum()),
                   "zips_with_polygon": int(master["has_polygon"].sum()), "address_rows": int(len(db)),
                   "sigungu": int(sum(len(v) for v in hier.values())), "emd": len(emd_out)},
        "simplify_m": SIMPLIFY_M, "coord_decimals": COORD_DEC, "grade_note": "이 파일의 값은 모두 주소(건물) 기준 집계이며 정확도 등급은 '정확'"}
dump(OUT / "meta.json", meta); print(json.dumps(meta["counts"], ensure_ascii=False))

## 7. 점검 엑셀 (한 파일, 시트별)

In [ ]:
chk = []
def add(name, value, rule, ok): chk.append({"점검": name, "값": value, "기준": rule, "판정": "통과" if ok else "확인 필요"})
for row in cmp_df.to_dict("records"):
    v = row["DB 기준 일치율"]
    add(f"{row['지역']}: DB 우편번호의 도형 보유율", v, f"≥ {TH['set_match_min']:.0%}", v >= TH["set_match_min"])
add("도형 ID 중복", int(q_dup), f"≤ {TH['dup_max']}", q_dup <= TH["dup_max"])
add("무효 도형", q_invalid, f"≤ {TH['invalid_max']}", q_invalid <= TH["invalid_max"])
add("겹침 쌍(>%d m²)" % OVERLAP_MIN_M2, len(overlaps), f"≤ {TH['overlap_pairs_max']}", len(overlaps) <= TH["overlap_pairs_max"])
area_bad = zone[zone["area_dev"] > TH["area_dev_max"]][["zip", "region", "SIG_KOR_NM", "BAS_AR", "area_km2_calc", "area_dev", "MVMN_DE", "MVMN_RESN"]].sort_values("area_dev", ascending=False)
add("속성면적(BAS_AR)과 도형면적이 %d%% 넘게 다른 구역 수" % (TH["area_dev_max"] * 100), len(area_bad), "= 0 (불일치 시 계산 면적 사용)", len(area_bad) == 0)
for k, v in hd_blank_rows.items():
    add(f"{k}: 행정동명 미표기 행 비율", v, "참고 (행정동 걸침에서 제외)", True)
add("건물관리번호 중복", dup_bldg, "= 0", dup_bldg == 0)
add("도형에만 있는 번호(주소 없음)", len(only_poly), "참고(원인 확인 필요)", True)
add("시군구 걸침 번호", int((summary["n_sigungu"] > 1).sum()), "참고", True)
add("읍면동 걸침 번호", int((summary["n_emd"] > 1).sum()), "참고", True)
add("행정동 걸침 번호", int((summary["n_hdong"] > 1).sum()), "참고", True)
verdict = pd.DataFrame(chk)

cross = []
for r in REGIONS:
    s = summary[summary["zip"].map(reg_of_zip) == r["key"]]
    cross.append({"지역": r["key"], "우편번호(주소 보유)": len(s), "시군구 걸침": int((s["n_sigungu"] > 1).sum()),
                  "읍면동 걸침": int((s["n_emd"] > 1).sum()), "행정동 걸침": int((s["n_hdong"] > 1).sum()),
                  "읍면동 걸침 비율": round(float((s["n_emd"] > 1).mean()), 4), "행정동 걸침 비율": round(float((s["n_hdong"] > 1).mean()), 4),
                  "번호당 주소 수 중앙값": float(s["n_addr"].median()), "번호당 주소 수 최대": int(s["n_addr"].max())})
cross_df = pd.DataFrame(cross)

limits = pd.DataFrame({"항목": [
    "집계 단위", "읍면동 이름", "비율의 의미", "도형 단순화", "도형만 있는 번호", "행정동 미표기", "면적 속성", "구 단위 도형", "범위주소 DB", "점검 기준값", "기준일"],
    "내용": [
    "건물(주소) 1건 = 1행. 건물 크기·세대수·인구는 반영되지 않는다(주소 건수 기준 비율).",
    "법정동명이 있으면 법정동, 없으면 읍면+리명. 행정동은 별도 열(행정동명)로 둔다.",
    "‘이 우편번호의 몇 %가 ○○동’은 주소 건수 비율이며 인구·면적 비율이 아니다.",
    f"EPSG:5179에서 {SIMPLIFY_M} m 단순화 후 좌표 {COORD_DEC}자리로 반올림. 이웃 경계에 미세한 틈·겹침이 생길 수 있다(위상 보존 단순화는 추후).",
    "우편번호 DB·범위주소·사서함·변경분 어디에도 주소가 없는 구역. 원인(공원·산지·신설 예정 등)은 확인하지 못했다.",
    "행정동명이 빈 행은 법정동과 같다고 가정하지 않고 '(미표기)'로 두었으며 행정동 걸침 계산에서 제외했다. 우체국 필드 명세에 설명이 없어 의미는 확인하지 못했다.",
    "도형의 면적 속성(BAS_AR)이 도형 면적과 크게 다른 구역이 있다(주로 2022년 이후 통합·경계 변경 구역). 원인은 확인하지 못했고 이 노트북은 도형에서 계산한 면적을 쓴다.",
    "도형의 시군구는 시 단위(예: 성남시)이며 구 단위는 우편번호 DB의 주소로만 알 수 있다.",
    "범위주소는 참고 자료이며 건물 단위 목록이 아니다. 여기서는 우편번호 집합 대조에만 사용했다.",
    "개수·겹침·면적 편차의 통과 기준은 임의의 1차 기준이며 실제 결과를 본 뒤 조정해야 한다.",
    f"우편번호 DB 고시 {NOTICE_DATE}(기준 {BASE_DATE}), 기초구역 도형 {ZONE_VER}."]})

files_df = pd.DataFrame([{"지역": k, "DB 파일 행 수": file_rows[k], "도형 파일": zone_files[k], "도형 파일 행 수": zone_rows[k]} for k in file_rows])
sum_out = master[["zip", "region", "sido", "primary_sigungu", "primary_sigungu_code", "primary_emd", "primary_emd_share",
                  "n_addr", "n_road", "n_sigungu", "n_emd", "n_hdong", "has_address", "has_polygon", "area_km2_calc", "BAS_AR", "MVMN_DE", "MVMN_RESN"]].rename(
    columns={"area_km2_calc": "면적_계산(km2)", "BAS_AR": "면적_도형속성(km2)"})
sigungu_df = (db.groupby(["시도", "sigungu_code", "sigungu"]).agg(주소수=("우편번호", "size"), 우편번호수=("우편번호", "nunique")).reset_index())
poly_q = zone[["zip", "region", "SIG_KOR_NM", "BAS_AR", "area_km2_calc", "area_dev", "MVMN_DE", "MVMN_RESN"]].copy()

xlsx = OUT / "qa" / "zipcode_qa_v1.xlsx"
with pd.ExcelWriter(xlsx, engine="openpyxl") as w:
    verdict.to_excel(w, sheet_name="종합판정", index=False); cmp_df.to_excel(w, sheet_name="개수_집합_대조", index=False)
    only_poly.to_excel(w, sheet_name="도형에만_있는_번호", index=False)
    pd.DataFrame({"DB에만_있는_번호": only_db}).to_excel(w, sheet_name="DB에만_있는_번호", index=False)
    cross_df.to_excel(w, sheet_name="계층_걸침", index=False); sigungu_df.to_excel(w, sheet_name="시군구별_요약", index=False)
    aux_df.to_excel(w, sheet_name="범위_사서함_변경분_대조", index=False); overlaps.to_excel(w, sheet_name="폴리곤_겹침", index=False)
    poly_q.to_excel(w, sheet_name="폴리곤_품질", index=False); area_bad.to_excel(w, sheet_name="면적_불일치", index=False); sum_out.to_excel(w, sheet_name="우편번호별_요약", index=False)
    files_df.to_excel(w, sheet_name="입력파일", index=False); limits.to_excel(w, sheet_name="데이터_한계", index=False)
    for ws in w.book.worksheets:
        ws.freeze_panes = "A2"
        for col in ws.columns:
            width = max(len(str(c.value)) if c.value is not None else 0 for c in list(col)[:200])
            ws.column_dimensions[col[0].column_letter].width = min(max(10, width * 1.6 + 2), 60)
print("엑셀:", xlsx, f"{xlsx.stat().st_size/1024:.0f} KB")
verdict